In [1]:
include("GenX.jl")
using .GenX
using HiGHS
using JuMP
using Gurobi
using DataFrames

[ Info: Running precompile script for GenX. This may take a few minutes.


  ____           __  __   _ _
 / ___| ___ _ __ \ \/ /  (_) |
| |  _ / _ \ '_ \ \  /   | | |
| |_| |  __/ | | |/  \ _ | | |
 \____|\___|_| |_/_/\_(_)/ |_|
                       |__/
 Version: 0.4.1


┌ Info: Running precompile script for GenX. This may take a few minutes.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\startup\genx_startup.jl:56


  ____           __  __   _ _
 / ___| ___ _ __ \ \/ /  (_) |
| |  _ / _ \ '_ \ \  /   | | |
| |_| |  __/ | | |/  \ _ | | |
 \____|\___|_| |_/_/\_(_)/ |_|
                       |__/
 Version: nothing


In [2]:
# Genx case_runners file,
case = "..\\example_systems\\1_three_zones"
settings_path = joinpath(case, "settings")
policies_path = joinpath(case, "policies")
output_folder = joinpath(case, "Results") # Write-output settings YAML file path,
genx_settings = joinpath(settings_path, "genx_settings.yml") # Settings YAML file path,
mysetup = GenX.configure_settings(genx_settings, output_folder) # mysetup dictionary stores settings and GenX-specific parameters,
optimizer = Gurobi.Optimizer
OPTIMIZER =  GenX.configure_solver(settings_path, optimizer)
myinputs =  GenX.load_inputs(mysetup, case)
EP =  GenX.generate_model(mysetup, myinputs, OPTIMIZER)
EP, solve_time =  GenX.solve_model(EP, mysetup)
myinputs["solve_time"] = solve_time
inputs = myinputs
setup = mysetup


Configuring Settings
Reading Input CSV Files
Network.csv Successfully Read!
Demand (load) data Successfully Read!
Fuels_data.csv Successfully Read!


┌ Info: Thermal.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Vre.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Storage.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Resource_minimum_capacity_requirement.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:727



Summary of resources loaded into the model:
-------------------------------------------------------
	Resource type 		Number of resources
	Thermal        		3
	VRE            		4
	Storage        		3
Total number of resources: 10
-------------------------------------------------------
Generators_variability.csv Successfully Read!
Validating time basis
Minimum_capacity_requirement.csv Successfully Read!
CO2_cap.csv Successfully Read!
CSV Files Successfully Read In From ..\example_systems\1_three_zones
Set parameter Username
Academic license - for non-commercial use only - expires 2025-08-28
Set parameter FeasibilityTol to value 1e-05
Set parameter PreDual to value 0
Set parameter Method to value 4
Set parameter TimeLimit to value 110000
Set parameter MIPGap to value 0.001
Set parameter OptimalityTol to value 1e-05
Set parameter AggFill to value 10
Set parameter Presolve to value 1
Discharge Module
Non-served Energy Module
Investment Discharge Module
Unit Commitment Module
Fuel Module
CO2 

Dict{Any, Any} with 43 entries:
  "HydrogenHourlyMatching"             => 0
  "NetworkExpansion"                   => 1
  "TimeDomainReductionFolder"          => "TDR_results"
  "WriteOutputs"                       => "full"
  "SystemFolder"                       => "system"
  "EnableJuMPStringNames"              => true
  "Trans_Loss_Segments"                => 1
  "ModelingtoGenerateAlternativeSlack" => 0.1
  "PlanningReserveMargin"              => 0
  "PoliciesFolder"                     => "policies"
  "MultiStage"                         => 0
  "ComputeConflicts"                   => 0
  "OverwriteResults"                   => 0
  "ObjScale"                           => 1
  "ModelingToGenerateAlternatives"     => 0
  "OutputFullTimeSeries"               => 1
  "MaxCapReq"                          => 0
  "MinCapReq"                          => 1
  "CO2Cap"                             => 2
  ⋮                                    => ⋮

In [26]:
Morris_range = load_dataframe(joinpath(case, "Method_of_morris_range.csv"))
groups = Morris_range[!, :Group]
p_steps = Morris_range[!, :p_steps]
total_num_trajectory = Morris_range[!, :total_num_trajectory][1]
num_trajectory = Morris_range[!, :num_trajectory][1]
len_design_mat = Morris_range[!, :len_design_mat][1]
uncertain_columns = unique(Morris_range[!, :Parameter])
#save_parameters = zeros(length(Morris_range[!,:Parameter]))
gen = inputs["RESOURCES"]
sigma = zeros((1, 2))


1×2 Matrix{Float64}:
 0.0  0.0

In [27]:
column = uncertain_columns[1]
col_sym = Symbol(lowercase(column))
# column_f is the function to get the value "column" for each generator
column_f = isdefined(GenX, col_sym) ? getfield(GenX, col_sym) :
           r -> getproperty(r, col_sym)

inv_cost_per_mwyr (generic function with 1 method)

In [29]:
column_f.(gen) .* (1 .+
               Morris_range[Morris_range[!, :Parameter] .== column, :Lower_bound] ./
               100)

10-element Vector{Float64}:
 58.86000000000001
 58.86000000000001
 58.86000000000001
 76.77
 87.48
 76.77
 87.48
 17.6256
 17.6256
 17.6256

In [30]:
sigma = [sigma;
             [column_f.(gen) .* (1 .+
               Morris_range[Morris_range[!, :Parameter] .== column, :Lower_bound] ./
               100) column_f.(gen) .*
                    (1 .+
                     Morris_range[Morris_range[!, :Parameter] .== column,
                 :Upper_bound] ./ 100)]]

11×2 Matrix{Float64}:
  0.0       0.0
 58.86     71.94
 58.86     71.94
 58.86     71.94
 76.77     93.83
 87.48    106.92
 76.77     93.83
 87.48    106.92
 17.6256   21.5424
 17.6256   21.5424
 17.6256   21.5424

In [6]:

for column in uncertain_columns
    col_sym = Symbol(lowercase(column))
    # column_f is the function to get the value "column" for each generator
    column_f = isdefined(GenX, col_sym) ? getfield(GenX, col_sym) :
               r -> getproperty(r, col_sym)
    sigma = [sigma;
             [column_f.(gen) .* (1 .+
               Morris_range[Morris_range[!, :Parameter] .== column, :Lower_bound] ./
               100) column_f.(gen) .*
                    (1 .+
                     Morris_range[Morris_range[!, :Parameter] .== column,
                 :Upper_bound] ./ 100)]]
end

DimensionMismatch: DimensionMismatch: arrays could not be broadcast to a common size; got a dimension with lengths 10 and 5

In [5]:

sigma = sigma[2:end, :]

p_range = mapslices(x -> [x], sigma, dims = 2)[:]

DimensionMismatch: DimensionMismatch: arrays could not be broadcast to a common size; got a dimension with lengths 10 and 5

In [7]:
df_duals = DataFrame()
T = inputs["T"]

if inputs["ro_settings"]["MarketBuyPrices"] == 1
    Z = inputs["Z"] 
    MZ = inputs["MZ"]
    GenX.write_ro_market_buy(joinpath(case, "resxx"), inputs, setup, EP)
    for i in 1:Z
        if i in MZ
            df_duals[!, "S_MarketBuy_$(i)"] =  vec(dual.(EP[:cDualSmb][i,:]).data)
        else
            df_duals[!, "S_MarketBuy_$(i)"] = zeros(T)
        end
    end
end

In [ ]:
using DataFrames

# Create a sample DataFrame
df = DataFrame(A = 1:10, B = 10:19)

# Create a new column with fewer values
new_column = [100, 200, 300, 400, 500]

# Add the new column to the DataFrame
df.C = [new_column; fill(missing, nrow(df) - length(new_column))]

# If you want to replace missing values with a specific value (e.g., 0):
# df.C = coalesce.(df.C, 0)

println(df)